In [ ]:
# notebooks/01_tax_optimization_demo.ipynb
# Run cells in order. Install deps first: pip install taxopt[notebook]

In [ ]:
# Cell 1 — Imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import time
import plotly.graph_objects as go
from datetime import date
from dataclasses import dataclass, replace
from typing import cast

from taxopt import (
    Portfolio,
    USCapitalGainsPolicy, LotMethod,
    CvxpyOptimizer, PortfolioPolicy, OptimizationInputs,
    TaxLedger,
)


In [ ]:
# Cell 2 — Download price data
# TICKERS: list[str] = [
#     "NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "AVGO", "GOOG", "META", "TSLA", "BRK-B",
#     "JPM", "LLY", "XOM", "JNJ", "WMT", "V", "MU", "COST", "MA", "NFLX",
#     "ABBV", "CVX", "PLTR", "PG", "HD", "CAT", "AMD", "GE", "BAC", "CSCO",
# ]
TICKERS: list[str] = [
    "VTI", "SPY", "QQQ", "SCHD", "IJR", "VUG", "VTV", "IVE", "IVW", "VO",
    "VEA", "VWO", "IEFA", "EEM", "VXUS", "EWJ", "EZU", "MCHI", "AGG", "BND",
    "BNDX", "SHY", "TLT", "LQD", "VNQ", "GLD", "DBC", "MLPA", "VGT", "XLV",
]
n: int = len(TICKERS)

_raw = yf.download(TICKERS, start="2015-01-01", end="2026-02-28", auto_adjust=True)
assert _raw is not None

raw: pd.DataFrame = cast(pd.DataFrame, _raw["Close"])[TICKERS].dropna()
returns: pd.DataFrame = cast(pd.DataFrame, raw.pct_change().dropna())

print(f"Price data: {raw.index[0].date()} → {raw.index[-1].date()}, {len(raw)} days")
raw.tail(3)


In [ ]:
# Cell 3 — Helper functions
def get_prices(as_of: date) -> dict[str, float]:
    ts = raw.index[raw.index <= pd.Timestamp(as_of)]
    row = raw.iloc[0] if len(ts) == 0 else raw.loc[ts[-1]]
    return {t: float(row[t]) for t in TICKERS}


def get_inputs(
    as_of: date,
    policy: PortfolioPolicy,
    lookback_days: int = 252,
) -> OptimizationInputs:
    end  = pd.Timestamp(as_of)
    rets = returns.loc[end - pd.Timedelta(days=lookback_days * 2) : end].dropna(how="any")
    if len(rets) < lookback_days:
        raise ValueError(f"Insufficient history on {as_of}: {len(rets)} days")
    win = rets.tail(lookback_days)

    cov = win.cov().to_numpy() * 252 + np.eye(n) * 1e-6  # annualized covariance with small diagonal regularization

    # Alpha: 12-1 month momentum → rank → z-score → beta-neutralize → risk-normalize
    momentum = (1 + rets.iloc[-252:-20]).prod() - 1
    ranked   = momentum.rank().to_numpy(dtype=float)
    w        = (ranked - ranked.mean()) / (ranked.std() + 1e-8)  # standardized signal

    b = _compute_betas(win)

    # scalar OLS coefficient a for regression w = a * b + residual
    num = float(w @ b)
    den = float(b @ b) if float(b @ b) > 1e-8 else 1.0
    v   = w - (num / den) * b  # residual weights, now market-beta-neutral

    v /= np.sqrt(max(float(v @ cov @ v), 1e-8))
    alpha = {t: float((cov @ v)[i]) for i, t in enumerate(TICKERS)}

    return OptimizationInputs(
        alpha=alpha,
        covariance=cov,
        assets=TICKERS,
        prices=get_prices(as_of),
        as_of=as_of,
        risk_aversion=policy.risk_aversion,
        tax_aversion=policy.tax_aversion,
        gross_leverage=policy.gross_leverage,
        net_exposure=policy.net_exposure,
        max_weight=policy.max_weight,
        max_turnover=policy.max_turnover,
    )


def _compute_betas(win: pd.DataFrame) -> np.ndarray:
    """Beta vs equal-weight index. Accepts pre-sliced window."""
    r_m  = win.mean(axis=1).to_numpy(dtype=float)
    r_mc = r_m - r_m.mean()
    denom = float((r_mc ** 2).sum())
    if denom < 1e-10:
        return np.zeros(len(win.columns))
    return np.array([
        float(((win[t].to_numpy() - win[t].mean()) * r_mc).sum()) / denom
        for t in win.columns
    ])


In [ ]:
# Cell 4 — rebalance loop with ledger
POLICY = USCapitalGainsPolicy(lot_method=LotMethod.MIN_GAIN)

# ── Portfolio policy ──────────────────────────────────────────
POLICY_OPT = PortfolioPolicy(
    risk_aversion  = 2.0,
    tax_aversion   = 1.0,
    gross_leverage = 4.0,   # L + S  (e.g. L130/S30 → 1.6)
    net_exposure   = 0.0,   # L - S
    max_weight     = 0.25,
    max_turnover   = 0.50,  # None = unlimited
)

# ── Solver settings ───────────────────────────────────────────
SOLVER = CvxpyOptimizer(
    solver="SCIP",
    verbose=False,
    tax_aware=True,                 # enable tax-aware optimization (enable objective term)
    relax_turnover=True,            # gradually widens turnover if infeasible
    turnover_relax_step=0.05,       # +5pp per attempt
    turnover_relax_max_attempts=5,  # up to +25pp before giving up
    mip_gap=0.01,                   # 1% MIP gap for faster solve at the cost of optimality guarantee
)

# All month-ends in range; rebalance on each except the last,
# which serves only as the EOM date of the final holding period.
all_dates: list[date] = pd.date_range("2017-12-30", "2026-02-28", freq="ME").date.tolist()
rebal_dates = all_dates[:-1]
eom_dates   = all_dates[1:]

portfolio = Portfolio(cash=100_000.0)
ledger    = TaxLedger()

# Synthetic first row: initial cash at the first rebal date (before any trades)
history: list[dict] = [{
    "date":          rebal_dates[0],
    "nav_eom":       portfolio.cash,
    "after_tax_nav": portfolio.cash,
    "st_realized":   0.0,
    "lt_realized":   0.0,
    "st_cumulative": 0.0,
    "lt_cumulative": 0.0,
    "tax_alpha":     0.0,
    "num_actions":   0,
    "num_lots":      0,
}]

for i, (rebal_date, eom_date) in enumerate(zip(rebal_dates, eom_dates)):
    is_first   = (i == 0)

    # --- Prices and NAV at the start of the period (BOM / previous EOM) ---
    prices_bom = get_prices(rebal_date)
    tv_pre     = portfolio.total_value(prices_bom)

    # --- Optimization inputs ---
    inputs = get_inputs(
        rebal_date,
        policy=replace(POLICY_OPT, max_turnover=None if is_first else POLICY_OPT.max_turnover),
    )

    # --- Optimize and apply trades at current month-end prices ---
    t0     = time.perf_counter()
    result = SOLVER.solve(portfolio, inputs, POLICY, tv_pre)
    elapsed = time.perf_counter() - t0

    if result.status not in ("optimal", "optimal_inaccurate"):
        print(f"{rebal_date}: {result.status}, skipping")
        continue

    portfolio, tax_report = portfolio.apply_actions(result.actions, prices_bom, POLICY, rebal_date)
    tv_post = portfolio.total_value(prices_bom)
    assert abs(tv_post - tv_pre) / tv_pre < 1e-4, f"NAV not conserved: {tv_pre:.2f} → {tv_post:.2f}"

    # --- Tax alpha this period: tax saving as fraction of NAV ---
    period_tax_impact = (
        tax_report.totals_by_type.get("short_term", 0.0) * POLICY.st_rate +
        tax_report.totals_by_type.get("long_term",  0.0) * POLICY.lt_rate
    )
    tax_alpha = -period_tax_impact / tv_pre

    # --- EOM of this holding period ---
    prices_eom = get_prices(eom_date)
    tv_eom     = portfolio.total_value(prices_eom)

    # --- Record realized gains into ledger AFTER computing period tax_alpha ---
    ledger.record(tax_report)
    at_nav_eom = ledger.after_tax_nav(portfolio, prices_eom, eom_date, POLICY)

    # --- Append row indexed by the EOM date ---
    history.append({
        "date":          eom_date,
        "nav_eom":       tv_eom,
        "after_tax_nav": at_nav_eom,
        "st_realized":   tax_report.totals_by_type.get("short_term", 0.0),
        "lt_realized":   tax_report.totals_by_type.get("long_term",  0.0),
        "st_cumulative": ledger.st_realized,
        "lt_cumulative": ledger.lt_realized,
        "tax_alpha":     tax_alpha,
        "num_actions":   len(result.actions),
        "num_lots":      sum(len(v) for v in portfolio.lots.values()),
        "solve_time":    elapsed,
        "eff_turnover":  result.effective_turnover,
    })

    to_str = f"{result.effective_turnover:.0%}"
    turnover_note = (
        f" [relaxed → cap={result.effective_turnover:.0%}]"
        if (inputs.max_turnover is not None
            and result.effective_turnover > inputs.max_turnover + 1e-4)
        else ""
    )
    print(
        f"{eom_date} EOM=${tv_eom:>10,.0f} AT-NAV=${at_nav_eom:>10,.0f} "
        f"ST={tax_report.totals_by_type.get('short_term', 0.0):>+8,.0f} "
        f"LT={tax_report.totals_by_type.get('long_term',  0.0):>+8,.0f} "
        f"TaxAlpha={tax_alpha*100:>+6.3f}% "
        f"TO={to_str:>4} "
        f"[{elapsed:.1f}s]{turnover_note}"
    )

df = pd.DataFrame(history).set_index("date")


In [ ]:
# Cell 5 — Summary metrics
final_nav       = float(df["after_tax_nav"].iloc[-1])
pretax_nav      = float(df["nav_eom"].iloc[-1])
initial_nav     = float(df["nav_eom"].iloc[0])
total_st        = float(df["st_realized"].sum())
total_lt        = float(df["lt_realized"].sum())
total_realized  = total_st + total_lt
cum_tax         = ledger.cumulative_tax_value(POLICY)
pretax_return   = (pretax_nav  / initial_nav - 1) * 100
aftertax_return = (final_nav   / initial_nav - 1) * 100
tax_drag        = pretax_return - aftertax_return
tax_efficiency  = aftertax_return / pretax_return if pretax_return != 0 else float("nan")
tax_alpha_ann   = float(np.prod(df["tax_alpha"].to_numpy(dtype=float) + 1)) ** (12 / len(rebal_dates)) - 1
n_years         = len(rebal_dates) / 12
pretax_cagr     = (pretax_nav  / initial_nav) ** (1 / n_years) - 1   # ← annualized
aftertax_cagr   = (final_nav   / initial_nav) ** (1 / n_years) - 1   # ← annualized

print("=" * 52)
print(f"  Backtest: {rebal_dates[0]} → {rebal_dates[-1]}  ({n_years:.1f} yrs)")
print("=" * 52)
print(f"  {'Initial NAV:':<32} ${initial_nav:>10,.0f}")
print(f"  {'Pre-tax final NAV:':<32} ${pretax_nav:>10,.0f}")
print(f"  {'After-tax final NAV:':<32} ${final_nav:>10,.0f}")
print()
print(f"  {'Pre-tax return:':<32} {pretax_return:>10.2f}%")
print(f"  {'Pre-tax CAGR:':<32} {pretax_cagr*100:>10.2f}%")   # ← added
print(f"  {'After-tax return:':<32} {aftertax_return:>10.2f}%")
print(f"  {'After-tax CAGR:':<32} {aftertax_cagr*100:>10.2f}%")  # ← added
print(f"  {'Tax drag:':<32} {tax_drag:>10.2f}%")
print(f"  {'Tax efficiency ratio:':<32} {tax_efficiency:>10.2%}")
print(f"  {'Annualized tax alpha:':<32} {tax_alpha_ann*100:>10.3f}%")
print()
print(f"  {'Cumul. ST realized:':<32} ${total_st:>10,.0f}")
print(f"  {'Cumul. LT realized:':<32} ${total_lt:>10,.0f}")
print(f"  {'Cumul. net realized:':<32} ${total_realized:>10,.0f}")
print(f"  {'Net tax impact (realized):':<32} ${cum_tax:>10,.0f}")
print(f"  {'  (neg = net tax saving)'}")
print()
print(f"  {'Effective rate on realized:':<32} {cum_tax/max(abs(total_realized),1)*100:>10.2f}%")
print(f"  {'Avg actions per rebalance:':<32} {df['num_actions'].mean():>10.1f}")
print(f"  {'Avg solve time:':<32} {df['solve_time'].mean():>10.2f}s")  # ← added since it's in df
print(f"  {'Final lot count:':<32} {int(df['num_lots'].iloc[-1]):>10d}")
print("=" * 52)

In [ ]:
# Cell 6 — Interactive charts
dates = df.index.astype(str).tolist()

# ── Chart 1: NAV vs After-Tax NAV ──────────────────────────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=dates, y=df["nav_eom"], name="Pre-Tax NAV",
                          line=dict(color="#3498db"), mode="lines+markers",
                          hovertemplate="%{x}<br>Pre-Tax NAV: $%{y:,.0f}<extra></extra>"))
fig1.add_trace(go.Scatter(x=dates, y=df["after_tax_nav"], name="After-Tax NAV",
                          line=dict(color="#2ecc71"), mode="lines+markers",
                          hovertemplate="%{x}<br>After-Tax NAV: $%{y:,.0f}<extra></extra>"))
fig1.update_layout(title="NAV vs After-Tax NAV (EOM)", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Value")
fig1.show()

# ── Chart 2: Realized ST / LT Gains per Rebalance ─────────────────────────
fig2 = go.Figure()
fig2.add_trace(go.Bar(x=dates, y=df["st_realized"], name="ST Realized",
                      marker_color="#e74c3c",
                      hovertemplate="%{x}<br>ST: $%{y:,.0f}<extra></extra>"))
fig2.add_trace(go.Bar(x=dates, y=df["lt_realized"], name="LT Realized",
                      marker_color="#2ecc71",
                      hovertemplate="%{x}<br>LT: $%{y:,.0f}<extra></extra>"))
fig2.update_layout(title="Realized ST / LT Gains per Rebalance",
                   barmode="group", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Gain / Loss")
fig2.show()

# ── Chart 3: Cumulative Realized ST / LT ──────────────────────────────────
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=dates, y=df["st_cumulative"], name="ST Cumulative",
                          line=dict(color="#e74c3c"), mode="lines+markers",
                          hovertemplate="%{x}<br>ST Cumul: $%{y:,.0f}<extra></extra>"))
fig3.add_trace(go.Scatter(x=dates, y=df["lt_cumulative"], name="LT Cumulative",
                          line=dict(color="#2ecc71"), mode="lines+markers",
                          hovertemplate="%{x}<br>LT Cumul: $%{y:,.0f}<extra></extra>"))
fig3.update_layout(title="Cumulative Realized ST / LT Gains", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Cumulative")
fig3.show()

# ── Chart 4: Cumulative Tax Alpha ──────────────────────────────────────────
cum_alpha = df["tax_alpha"].astype(float).cumsum() * 100
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=dates, y=cum_alpha, name="Cumul. Tax Alpha",
                          line=dict(color="#9b59b6"), mode="lines+markers",
                          fill="tozeroy", fillcolor="rgba(155,89,182,0.15)",
                          hovertemplate="%{x}<br>Tax Alpha: %{y:.3f}%<extra></extra>"))
fig4.add_hline(y=0, line_dash="dash", line_color="gray")
fig4.update_layout(title="Cumulative Tax Alpha (% of NAV)", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="% of NAV")
fig4.show()


In [ ]:
# Cell 7 — Lot inspection
final_date   = rebal_dates[-1]
final_prices = get_prices(final_date)

rows = [
    {
        "asset":       asset,
        "qty":         round(lot.quantity, 4),
        "basis":       round(lot.cost_basis, 2),
        "price":       round(final_prices[asset], 2),
        "unreal_gain": round((final_prices[asset] - lot.cost_basis) * lot.quantity, 2),
        "days_held":   (final_date - lot.acquisition_date).days,
        "gain_type": POLICY.classify_gain(lot, final_date, 0.0),
    }
    for asset, lots in portfolio.lots.items()
    for lot in lots
]
lot_df = pd.DataFrame(rows).sort_values("unreal_gain")
print(f"Total unrealized:  ${float(lot_df['unreal_gain'].sum()):,.0f}")
print(f"ST unrealized:     ${float(lot_df.loc[lot_df['gain_type']=='short_term','unreal_gain'].sum()):,.0f}")
print(f"LT unrealized:     ${float(lot_df.loc[lot_df['gain_type']=='long_term','unreal_gain'].sum()):,.0f}")
lot_df
